# 21 Sep

## wonko script to give entry accepted name and assign rest as synonyms

- am still keeping the origional scientific name so that I can match compounds and plants later

In [ ]:
import pandas as pd

merged_plants_file = ''
wcvp_file = '' 
error_file = ''
output_file = ''

df_merged = pd.read_csv(merged_plants_file)
df_wcvp = pd.read_csv(wcvp_file, sep='|')

df_merged['Unnamed: 0'] = df_merged['temp_id']
df_merged['acceptednameusageid'] = pd.to_numeric(df_merged['acceptednameusageid'],errors='coerce').astype('Int64')
df_wcvp['taxonid'] = pd.to_numeric(df_wcvp['taxonid'], errors='coerce').astype('Int64')


def get_properName(accepted_id):
    accepted_id = int(accepted_id)
    row = df_wcvp.loc[df_wcvp['taxonid'] == accepted_id]
    if row.empty:
        return None
    else:
        return row.iloc[0]

print('Begining official name assignment', flush=True)

entry = []
errors = []
count = 0
total = len(df_merged)
for idx,row in df_merged.iterrows():
    count += 1
    proper_id = row.acceptednameusageid

    if pd.notna(proper_id):
        proper_data = get_properName(proper_id)
        if proper_data is None:
            errors.append([row.original_scientificName, row.ID, 'ID Not found in wcvp apparently'])
            continue
        plant_id = row.temp_id
        accepted_id = proper_id
        name = proper_data['scientfiicname']
        family = proper_data['family']
        authorship = proper_data['scientfiicnameauthorship']
        namepublishedin = proper_data['namepublishedin']
        powo_id_accepted = proper_data['scientificnameid']
        origional_coconut_name = row.original_scientificName
        gbif_name = row.scientificName
        gbif_ID = row.ID 
        gbif_accepted_name = row.acceptedScientificName
        gbif_acceptef_id = row.acceptedID
        powo_scientific_name = row.scientfiicname
        powo_id = row.taxonid
        entry.append([plant_id, accepted_id, name, family, authorship, namepublishedin, powo_id_accepted, origional_coconut_name, gbif_name, gbif_ID, gbif_accepted_name, gbif_acceptef_id, powo_scientific_name, powo_id])
    else:
        errors.append([row.original_scientificName, row.ID, 'No acceepted usage name ID'])

    prog = count/total*100

    if count % 1000 == 0:
        print(f'Progress: {prog:.2f}%', flush=True)

cols = ['plant_id','accepted_id_wcvp', 'name', 'family', 'authorship', 'namepublishedin', 'powo_id_accepted', 'original_coconut_name', 'gbif_name', 'gbif_ID', 'gbif_accepted_name', 'gbif_accepted_id', 'powo_scientific_name', 'powo_id'] 
df_dups = pd.DataFrame(entry, columns= cols)

df_errors = pd.DataFrame(errors,columns=['original_scientificName', 'gbif_ID', 'error'])
df_errors.to_csv(error_file, index=False)

def join_dedup(things):
    out = []
    added = set()
    for thing in things:
        if pd.isna(thing):
            continue
        thing = str(thing).strip()
        if not thing or thing not in added:
            added.add(thing)
            out.append(thing)
    return '|'.join(out)  
        


out_df = (df_dups.groupby('accepted_id',sort=False,as_index=False).agg(join_dedup))
out_df.drop(columns=['plant_id'],inplace=True)
out_df['plant_id'] = [num for num in range(1,len(out_df)+1)]
out_df.insert(0,'plant_id',out_df.pop('plant_id'))
out_df.to_csv(output_file, index=False)

print(f'Input rows: {len(df_merged):,}')
print(f'Accepted-name rows retained: {len(df_dups):,}')
print(f'Accepted taxa output: {len(out_df):,}')
print(f'Errors: {len(df_errors):,}')



# 22 Sep
## checking out what came back from the server to see if it can be sent to the DB

Files used: coconut_full/coconut_plants/merged_gbif_wcvp.csv, wcvp_taxon.csv
files created: /home/school/masters/Scripts/coconut_full/coconut_plants/errors_prepping_plants_for_db.csv, /home/school/masters/Scripts/coconut_full/coconut_plants/coconut_plants_for_DB.csv

In [14]:
import pandas as pd

df_pre_dedup = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_plants/merged_gbif_wcvp.csv')
df_dedup = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_plants/coconut_plants_for_DB.csv')
df_errors = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_plants/errors_prepping_plants_for_db.csv')

tot_before = df_pre_dedup.shape[0]
tot_after = df_dedup.shape[0]
tot_errors = df_errors.shape[0]

tot_should_be = df_pre_dedup.drop_duplicates(subset='acceptednameusageid').shape[0]

print(f'The merge file before duplication: {tot_before:,}')
print(f'The end deduplicated file: {tot_after:,}')
print(f'The number of errors: {tot_errors:,}')
print(f'The number of entires there should have been (drop_dups on accpeptednameusageid): {tot_should_be:,}')
print(f'The number of entires removed by deduplication and errors: {tot_before-tot_after:,}')

The merge file before duplication: 284,276
The end deduplicated file: 54,453
The number of errors: 22,888
The number of entires there should have been (drop_dups on accpeptednameusageid): 54,454
The number of entires removed by deduplication and errors: 229,823


In [18]:
import pandas as pd

df_errors = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_plants/errors_prepping_plants_for_db.csv')

num_errorTypes = df_errors['error'].value_counts()
num_expected = df_pre_dedup['acceptednameusageid'].isna().sum()

print(f'{num_errorTypes}')
print(f'expect {num_expected:,}')

error
No acceepted usage name ID    22888
Name: count, dtype: int64
expect 22,888


In [22]:
import pandas as pd

df_pre_dedup = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_plants/merged_gbif_wcvp.csv')

not_in_wcvp = df_pre_dedup.drop_duplicates(subset='original_scientificName')
sum_deduped_missing_id = not_in_wcvp['acceptednameusageid'].isna().sum()

print(f'number of plants that missing accepted name usage ID deduplicated {sum_deduped_missing_id:,}')

number of plants that missing accepted name usage ID deduplicated 119


## Now I need to prep the plants into a DB datadump

In [37]:
import pandas as pd
import sys

df = pd.read_csv('/home/school/masters/Scripts/coconut_full/coconut_plants/coconut_plants_for_DB.csv')
families = pd.read_csv('/home/school/masters/Scripts/APG_IV/data-apg4-master/families.csv')

df.drop(columns = ['accepted_id_wcvp'], inplace=True)

df = df.loc[df['family'].ne('Pinaceae')].copy()
df = df.loc[df['family'].ne('Cupressaceae')].copy()
df = df.loc[df['family'].ne('Polypodiaceae')].copy()
df = df.loc[df['family'].ne('Aspleniaceae')].copy()
df = df.loc[df['family'].ne('Dennstaedtiaceae')].copy()


def get_famid(name):
    try:
        name = families['family_id'].loc[families['family'] == name].values[0]
        if name:
            return name
        else:
            sys.exit()
    except:
        print(name)
        sys.exit()


df['family'] = df['family'].apply(get_famid)

def collect_names(names,ids):
    if len(names) != len(ids):
        sys.exit()
    names = names.split('|')
    names = [name.strip() for name in names]
    ids = ids.split('|')
    ids = [id.strip() for id in ids]
    out = []
    for name,id in zip(names,ids):
        out.append(f'{name} ({id})')
    out = set(out, sort=False)
    return '|'.join(out) 
    
df['gbif_name'] =  df['gbif_name'].apply(lambda row: collect_names(df['gbif_name'],df['gbif_ID']), axis=1)
df.drop(columns=['gbif_ID'], inplace=True)

df['gbif_accepted_name'] =  df['gbif_accepted_name'].apply(lambda row: collect_names(df['gbif_accepted_name'],df['gbif_accepted_ID']), axis=1)
df.drop(columns=['gbif_accepted_ID'], inplace=True)

df.to_csv('/home/school/masters/Scripts/coconut_full/coconut_plants/coconut_plants_for_DB_with_family_id.csv', index=False)



Petiveriaceae


SystemExit: 

/home/school/masters/Scripts/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
